# पाठ 11 - Model Context Protocol (MCP)

**Model Context Protocol (MCP)** एक खुला मानक है जो एजेंटों को रनटाइम पर गतिशील रूप से टूल, संसाधन और डेटा स्रोत खोजने तथा उपयोग करने में सक्षम बनाता है। एजेंट में टूल्स को हार्डकोड करने के बजाय, MCP एजेंटों को उन बाहरी सर्वरों से कनेक्ट करने देता है जो मांग पर क्षमताएँ प्रदान करते हैं।

इस पाठ में, आप सीखेंगे:
- MCP क्या है और एजेंट प्रणालियों के लिए यह क्यों महत्वपूर्ण है
- MCP क्लाइंट-सर्वर आर्किटेक्चर कैसे काम करती है
- MCP-शैली की टूल खोज का उपयोग करने वाले एजेंट कैसे बनाएं


## सेटअप

**पूर्वापेक्षाएँ:**
- Azure AI Foundry परियोजना जिसमें एक तैनात मॉडल हो
- प्रमाणीकरण के लिए `az login` चलाएँ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Model Context Protocol (MCP) क्या है?

MCP एआई एजेंट्स के लिए बाहरी टूल्स और डेटा स्रोतों की खोज करने और उनके साथ इंटरैक्ट करने का एक मानक तरीका परिभाषित करता है:

- **MCP Server**: टूल्स, संसाधन और प्रॉम्प्ट्स को एक मानक प्रोटोकॉल के माध्यम से उजागर करता है
- **MCP Client**: एजेंट रनटाइम जो सर्वरों से जुड़ता है और उपलब्ध क्षमताओं का पता लगाता है
- **Dynamic Discovery**: एजेंटों को हार्डकोडेड टूल्स की आवश्यकता नहीं होती — वे रनटाइम पर क्या उपलब्ध है इसका पता लगाते हैं

यह उन विस्तार-योग्य एजेंट सिस्टमों के निर्माण के लिए शक्तिशाली है जहाँ नई क्षमताएँ एजेंट कोड को बदले बिना जोड़ी जा सकती हैं।


## MCP कैसे काम करता है

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. एजेंट (MCP क्लाइंट) एक MCP सर्वर से जुड़ता है
2. सर्वर उपलब्ध टूल और उनके स्कीमा की सूची के साथ प्रतिक्रिया करता है
3. तब एजेंट अपनी तर्क प्रक्रिया के दौरान किसी भी खोजे गए टूल को कॉल कर सकता है
4. परिणाम उसी प्रोटोकॉल के माध्यम से वापस लौटते हैं


## MCP टूल खोज का अनुकरण

एक वास्तविक MCP सर्वर के लिए एक चल रहे सर्वर प्रोसेस की आवश्यकता होती है, इसलिए हम इस पैटर्न को `@tool` फ़ंक्शनों का उपयोग करके प्रदर्शित करेंगे जो MCP-से जुड़ी आवास सेवा द्वारा प्रदान की जाने वाली कार्यक्षमता का अनुकरण करते हैं।

प्रोडक्शन में, ये टूल स्थानीय रूप से परिभाषित करने के बजाय डायनामिक रूप से एक MCP सर्वर से खोजे जाएंगे।


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-शैली के उपकरणों के साथ एक एजेंट बनाना


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP उत्पादन में

उत्पादन पर्यावरण में, MCP शक्तिशाली पैटर्न सक्षम करता है:

- **डायनामिक उपकरण खोज**: एजेंट्स MCP सर्वरों से कनेक्ट होते हैं और रनटाइम पर उपकरणों की खोज करते हैं
- **विच्छेदित आर्किटेक्चर**: उपकरण प्रदाता एजेंट से स्वतंत्र रूप से अपडेट कर सकते हैं
- **संगठनों के पार साझा करना**: टीमें MCP सर्वरों के माध्यम से क्षमताएँ उपलब्ध करा सकती हैं जिन्हें कोई भी एजेंट उपयोग कर सकता है
- **Microsoft Agent Framework समर्थन**: MAF में `mcp` इंटीग्रेशन के माध्यम से बिल्ट-इन MCP क्लाइंट समर्थन है

MAF के साथ एक वास्तविक MCP सर्वर का उपयोग करने के लिए, आप `hosted_mcp_tool()` या MCP क्लाइंट इंटीग्रेशन के माध्यम से कनेक्ट करेंगे।

**और जानें:**
- [MCP विनिर्देश](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP समर्थन](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## सारांश

इस पाठ में, आपने सीखा:
- **MCP** एजेंटों और टूल प्रदाताओं के बीच डायनेमिक टूल खोज के लिए एक खुला मानक है
- **क्लाइंट-सर्वर आर्किटेक्चर** एजेंटों को रनटाइम पर क्षमताओं की खोज करने देता है
- MCP सक्षम करता है **विस्तारयोग्य, पृथक एजेंट सिस्टम** जहाँ टूलों को कोड परिवर्तन किए बिना जोड़ा जा सकता है
- Microsoft Agent Framework उत्पादन उपयोग के लिए **इन-बिल्ट MCP समर्थन** प्रदान करता है


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
यह दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके अनुवादित किया गया है। हालांकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान रखें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ को उसकी मूल भाषा में अधिकारिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न होने वाली किसी भी गलतफहमी या गलत व्याख्या के लिए हम जिम्मेदार नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
